<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgb(234, 62, 131);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ sample routes generation ★ </h3>
  <p> The code in this notebook finds and plots a set of paths from origin to destination based on varying weight adjustment functions.</p>
</div>

In [84]:
import webcolors as wc
import osmnx as ox
import os
import builtins
import networkx as nx
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely import wkt
from shapely.geometry import box
import folium
import warnings
warnings.filterwarnings("ignore")
import csv
from IPython.display import display
from html2image import Html2Image
from PIL import Image, ImageDraw, ImageFont
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time
import tempfile
import math
import folium
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import unary_union

<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgb(234, 62, 131);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ can modify ★ </h3>
  <p> 1. Change {curr_score} to be the name of the column holding your scores.</p>
  <p> 2. routes_df holds origin/destination pairs (lat, lon format).</p>
  <p> 3. adjusted_edges holds edges with scores per edge.</p>
  <p> 4. north, east, sout, west define the boundary box. </p>
  <p> 5. Note: appeal score is 1 as a placeholder.</p>
</div>

In [85]:

routes_df = pd.read_csv('../data/routes.csv')
adjusted_edges = pd.read_csv('../all_scored_edges_filtered.csv')
adjusted_edges['appeal_score'] = 1.0
north, east, south, west = 47.62425802431359, -122.35919327219091, 47.596252480947555, -122.31765250756945
curr_score = 'access_score'


<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgb(234, 62, 131);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ do not modify ★ </h3>
</div>

In [86]:
def assign_user_weight(df, wts: dict, isday):
    if isday:
        df['weighted_score'] = df['access_score'] * wts['access_score'] + df['safety_score_day_buffer'] * wts['safety_score_day'] + df['appeal_score'] * wts['appeal_score']
        return df
    df['weighted_score'] = df[curr_score] * wts['access_score'] + df['safety_score_night_buffer'] * wts['safety_score_night'] + df['appeal_score'] * wts['appeal_score']
    return df




<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgb(234, 62, 131);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ can modify ★ </h3>
  <p>1. Define score/column weights in the first cell.</p>
  <p> 2. Define columns for each function weight in the second cell - also fill out fxn_cols and fxn_names accordingly (lengths must match).</p>
</div>

In [87]:
adjusted_edges = assign_user_weight(adjusted_edges, {'access_score' : 1.0, 'safety_score_night': 0, 'appeal_score': 0}, False)
adjusted_edges['normal_score'] = 1 - adjusted_edges['weighted_score']
cost_score = 'normal_score'

In [88]:
adjusted_edges['inverse'] = adjusted_edges['weighted_score'].apply(lambda x: 1 / x if x != 0 else 1.0)
adjusted_edges['inverse_log'] = adjusted_edges['weighted_score'].apply(lambda x: 1 / np.log(x+1) if x!= 0 else 1.0)
adjusted_edges['inverse_log_nat'] = adjusted_edges['weighted_score'].apply(lambda x: 1 / math.log(x+1, math.e) if x > 0 else 1.0)
adjusted_edges['inverse_sqrt'] = adjusted_edges['weighted_score'].apply(lambda x: 1 / np.sqrt(x) if x != 0 else 1.0)
adjusted_edges['inverse_sqr'] = adjusted_edges['weighted_score'].apply(lambda x: 1 / x**2 if x != 0 else 1.0)
adjusted_edges['inverse_exp'] = adjusted_edges['weighted_score'].apply(lambda x: 1 / 2 **x if x != 0 else 1.0)

adjusted_edges['linear1'] = 1 - adjusted_edges['weighted_score']
adjusted_edges['linear1p4'] = 1.4 - adjusted_edges['weighted_score']
adjusted_edges['linear10'] = 10 - adjusted_edges['weighted_score']
new_score = 'linear1'

In [89]:
fxn_cols = ['linear1', 'linear1p4', 'linear10', 'inverse', 'inverse_log', 'inverse_log_nat', 'inverse_sqrt', 'inverse_exp', 'length']
fxn_names = ['Linear 1', 'Linear 1.4', 'Linear 10', 'Inverse', 'Inverse Log', 'Shortest Path']

num_fxns = len(fxn_cols)

In [90]:
colors = [
        'orangered', 'dodgerblue', 'darkorange', 'darkviolet', 'forestgreen',
        'magenta', 'black', 'sienna', 'lightseagreen', 'mediumblue'
    ]

color_nns = [
        'Red', 'Blue', 'Orange', 'Purple', 'Green',
        'Pink', 'Black', 'Brown', 'Turquoise', 'Dark Blue']

<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgb(234, 62, 131);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ do not modify ★ </h3>
  <p> Prepare graphs and routing - only run once for whole process.</p>
</div>

In [91]:
def prepare_graphs():
    graph = ox.graph_from_bbox(bbox=(north, south, east, west), network_type = 'walk')
    nodes, edges = ox.graph_to_gdfs(graph)
    adjusted_edges['geometry'] = adjusted_edges['geometry'].apply(wkt.loads)

    default_speed = 4800
    adjusted_edges['travel_time'] = (adjusted_edges['length'] / default_speed) * 60
    modified_edges = gpd.GeoDataFrame(adjusted_edges, crs='epsg:4326')

    # assuming 'modified_edges' has 'u', 'v', and 'key' columns
    modified_edges.set_index(['u', 'v', 'key'], inplace=True)

    G_2 = ox.load_graphml(filepath='../data/seattle_walk.graphml')

    G = ox.utils_graph.graph_from_gdfs(nodes, modified_edges)

    def is_footway(highway_val):
        if highway_val is None:
            return False
        if isinstance(highway_val, (list, tuple, set)):
            return 'footway' in highway_val
        return highway_val == 'footway'

    foot_edges = [
        (u, v, k)
        for u, v, k, data in G_2.edges(keys=True, data=True)
        if is_footway(data.get('highway'))
    ]
    G_2 = G_2.edge_subgraph(foot_edges).copy()

    # (Optional) keep only the largest connected component for cleanliness
    # For pedestrian networks, weakly connected usually makes sense:
    G_2 = ox.utils_graph.get_largest_component(G_2, strongly=False)

    return G, G_2



In [92]:
# Construct of list lines from a list of nodes
def node_list_to_path(G, node_list):
    """
    Given a list of nodes, return a list of lines that together
    follow the path
    defined by the list of nodes.
    Parameters
    ----------
    G : networkx multidigraph
    route : list
        the route as a list of nodes
    Returns
    -------
    lines : list of lines given as pairs ( (x_start, y_start),
    (x_stop, y_stop) )
    """
    edge_nodes = list(zip(node_list[:-1], node_list[1:]))
    lines = []
    for u, v in edge_nodes:
        # if there are parallel edges, select the shortest in length
        data = min(G.get_edge_data(u, v).values(),
                   key=lambda x: x[new_score])
        # if it has a geometry attribute
        if 'geometry' in data:
            # add them to the list of lines to plot
            xs, ys = data['geometry'].xy
            lines.append(list(zip(xs, ys)))
        else:
            # if it doesn't have a geometry attribute,
            # then the edge is a straight line from node to node
            x1 = G.nodes[u]['x']
            y1 = G.nodes[u]['y']
            x2 = G.nodes[v]['x']
            y2 = G.nodes[v]['y']
            line = [(x1, y1), (x2, y2)]
            lines.append(line)
    return lines

In [93]:
# Find the coordinates of for series of points along the path
def prepare_path_coordinates(G, path):
    lines = node_list_to_path(G, path)
    lat, long = [], []
    for line in lines:
        for x, y in line:
            long.append(x)
            lat.append(y)
    return lat, long

In [94]:
# Retrieve the original score based on the matching edge
def retrieve_score(adjusted_edges, u, v, new_score):
    match = adjusted_edges[
        ((adjusted_edges['u'] == u) & (adjusted_edges['v'] == v))
    ]
    if not match.empty:
        return match[new_score].iloc[0]
    else:
        raise ValueError(f"No score found for edge({u}, {v})")



<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgb(234, 62, 131);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ do not modify ★ </h3>
  <p> Calculate per-path scores and other information.</p>
</div>

In [95]:
def calculate_path_score(G, path):
    normal_score = 0
    segments = 0
    for i, node in enumerate(path[:-1]):
        u, v = node, path[i+1]
        try:
            normal_score += (1-retrieve_score(adjusted_edges, u, v, 'weighted_score')) # the function inverse scores where smaller better
            segments += 1
        except ValueError as e:
            print(e)
            continue
    normal_score_avg = normal_score / segments
    return normal_score_avg

In [96]:
# Calculate travel time for a path
def calculate_path_travel_time(G, path):
    total_travel_time = 0
    edge_nodes = list(zip(path[:-1], path[1:]))
    for u, v in edge_nodes:
        edge_data = G.get_edge_data(u, v)
        if edge_data and 0 in edge_data:
            travel_time = edge_data[0].get('travel_time', 0)
        total_travel_time += travel_time    
    return total_travel_time

In [97]:
# Calculate path length
def calculate_path_length(G, path):
    total_length = 0
    edge_nodes = list(zip(path[:-1], path[1:]))
    
    for u, v in edge_nodes:
        edge_data = G.get_edge_data(u, v)
        if edge_data and 0 in edge_data:
            length = edge_data[0].get('length', 0)
        total_length += length
    return total_length

<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgb(234, 62, 131);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ do not modify ★ </h3>
  <p> Functions for plotting paths and saving images/tables/summaries to respective files. All table info is all stored as csv format.</p>
</div>

In [98]:
def plot_all_paths_in_list_with_labels_2(
    G,
    paths,
    origin_point,
    destination_point,
    id,
    path_names_coded=fxn_cols,
    path_names=fxn_names,
    path_colors=colors,
    path_colors_nicknames=color_nns,
    tiles="cartodbpositron",
    show_segment_labels=True,
    score_label_fmt="{:.2f}",
    show_context_edges=True,
    context_corridor_m=40,        # selection radius (meters) around each path
    context_color="#666666",
    context_weight=3,
    context_opacity=0.2,
    context_label_px=12,
):
    
    

    # ---------- helpers ----------
    def _edge_geom_from_G(u, v):
        """Pick the (u,v) edge used for routing (min by new_score) and return a LineString geometry."""
        data = G.get_edge_data(u, v)
        if not data:
            return None
        best = min(data.values(), key=lambda x: x.get(new_score, float("inf")))
        if best.get("geometry") is not None:
            return best["geometry"]
        # fallback: straight line
        x1, y1 = G.nodes[u]["x"], G.nodes[u]["y"]
        x2, y2 = G.nodes[v]["x"], G.nodes[v]["y"]
        return LineString([(x1, y1), (x2, y2)])

    def _draw_segment_with_label(m, geom, color, tooltip_html, label_text, is_context=False):
        if geom is None:
            return
        coords = [(lat, lon) for lon, lat in geom.coords]  # (x,y)->(lon,lat)->(lat,lon)
        folium.PolyLine(
            locations=coords,
            color=color,
            weight=(context_weight if is_context else 5),
            opacity=(context_opacity if is_context else 0.95),
            tooltip=folium.Tooltip(tooltip_html, sticky=True),
        ).add_to(m)
        if show_segment_labels:
            try:
                midpoint = geom.interpolate(0.5, normalized=True)
                px = context_label_px if is_context else context_label_px
                folium.Marker(
                    location=[midpoint.y, midpoint.x],
                    icon=folium.DivIcon(html=(
                        f'<div style="display:inline-block;white-space:nowrap;'
                        f'font-size:{px}px;background-color:white;padding:2px 4px;'
                        f'text-align:center;line-height:1;border:1px solid #ccc;'
                        f'border-radius:3px;max-width:60px;">{label_text}</div>'
                    ))
                ).add_to(m)
            except Exception:
                pass

    def _uv_key(a, b):
        return tuple(sorted((int(a), int(b))))

    # ---------- build robust undirected edge lookup (once) ----------
    edges_gdf = adjusted_edges
    if not isinstance(edges_gdf, gpd.GeoDataFrame):
        edges_gdf = gpd.GeoDataFrame(edges_gdf, geometry="geometry", crs="EPSG:4326")

    ae = edges_gdf.copy()

    # Normalize u,v types so they match graph node ids
    for col in ["u", "v"]:
        ae[col] = pd.to_numeric(ae[col], errors="coerce")
    # Cast floats that are integral to ints (Int64 to allow NA)
    for col in ["u", "v"]:
        if np.issubdtype(ae[col].dtype, np.number):
            ae[col] = ae[col].round().astype("Int64")

    ae = ae.dropna(subset=["u", "v"]).copy()
    ae["uv_key"] = ae.apply(lambda r: _uv_key(r["u"], r["v"]), axis=1)

    # choose the row per undirected pair that routing would choose (min by new_score)
    if new_score in ae.columns:
        best_idx = ae.groupby("uv_key")[new_score].apply(lambda s: s.astype(float).idxmin())
        best = ae.loc[best_idx]
    else:
        best = ae.sort_values("uv_key").drop_duplicates("uv_key", keep="first")

    # Build lookup dict
    edge_lookup = {}
    for _, row in best.iterrows():
        k = row["uv_key"]
        edge_lookup[k] = {
            "osm_id": row["osm_id"] if "osm_id" in row and pd.notnull(row["osm_id"]) else "N/A",
            "length": float(row["length"]) if "length" in row and pd.notnull(row["length"]) else np.nan,
            "score": float(row[cost_score]) if cost_score in row and pd.notnull(row[cost_score]) else np.nan,
            "geometry": row["geometry"] if "geometry" in row else None,
        }

    # ---------- base map ----------
    lat_center = np.mean([origin_point[0], destination_point[0]])
    lon_center = np.mean([origin_point[1], destination_point[1]])
    m = folium.Map(location=[lat_center, lon_center], zoom_start=14, tiles=tiles)
    folium.Marker(origin_point, popup="Origin", icon=folium.Icon(color='red')).add_to(m)
    folium.Marker(destination_point, popup="Destination", icon=folium.Icon(color='green')).add_to(m)

    drawn_context_ids = set()


    for i, path_info in enumerate(paths):
        path = path_info['paths']
        # ---- path-level stats ----
        normal_score = calculate_path_score(G, path)
        length = calculate_path_length(G, path)
        time = calculate_path_travel_time(G, path)

        popup_text = (
            f"Path {path_info['function']} ({path_colors_nicknames[i % len(path_colors)]})"
            f"<br>Distance: {length:.2f} m"
            f"<br>Time: {time:.2f} min"
            f"<br>Normalized Score: {normal_score:.2f}"
        )
        seg_color = path_colors[i % len(path_colors)]

        # faint backbone
        lat, lon = prepare_path_coordinates(G, path)
        coords = list(zip(lat, lon))
        folium.PolyLine(coords, color=seg_color, weight=5, opacity=0.35,
                        popup=folium.Popup(popup_text, max_width=320)).add_to(m)

        # ---- on-path edges (with labels) ----
        path_geoms = []
        path_pairs = set()
        for u, v in zip(path[:-1], path[1:]):
            geom = _edge_geom_from_G(u, v)
            if geom is not None:
                path_geoms.append(geom)
            path_pairs.add((u, v))

            # get label score from lookup (exact row used for routing)
            info = edge_lookup.get(_uv_key(u, v))
            if info:
                score_val = info["score"]
                score_text = score_label_fmt.format(score_val) if np.isfinite(score_val) else "N/A"
                osm_id = info["osm_id"]
                length_text = f"{info['length']:.1f}" if np.isfinite(info["length"]) else "N/A"
            else:
                # fallback via retrieve_score if not found in lookup (should be rare)
                score_text = "N/A"
                try:
                    s_val = retrieve_score(adjusted_edges, u, v, cost_score)
                    if s_val is not None and np.isfinite(float(s_val)):
                        score_text = score_label_fmt.format(float(s_val))
                except Exception:
                    pass
                osm_id, length_text = "N/A", "N/A"

            tooltip_html = f"""
            <b>OSM ID:</b> {osm_id}<br>
            <b>Score:</b> {score_text}<br>
            <b>Length (m):</b> {length_text}<br>
            """
            _draw_segment_with_label(m, geom, seg_color, tooltip_html, score_text, is_context=False)

        # ---- nearby/context edges (optional) ----
        if show_context_edges and path_geoms:
            # selection only; we do NOT draw the corridor itself
            gs = gpd.GeoSeries(path_geoms, crs="EPSG:4326").to_crs(3857)
            corridor = unary_union(list(gs)).buffer(context_corridor_m)
            corridor_ll = gpd.GeoSeries([corridor], crs=3857).to_crs(4326).iloc[0]

            cand = edges_gdf[edges_gdf.geometry.intersects(corridor_ll)].copy()

            # drop edges that are exactly on the path (both directions)
            if {"u", "v"}.issubset(cand.columns):
                on_path = {(min(u, v), max(u, v)) for (u, v) in path_pairs}  # undirected set
                cand["uv_key"] = cand.apply(lambda r: _uv_key(r["u"], r["v"]), axis=1)
                cand = cand[~cand["uv_key"].isin(on_path)]

            for idx, r in cand.iterrows():
                if idx in drawn_context_ids:
                    continue
                geom_ctx = r.geometry
                if geom_ctx is None:
                    continue

                info = edge_lookup.get(_uv_key(r.get("u"), r.get("v")))
                if info:
                    score_val = info["score"]
                    score_text = score_label_fmt.format(score_val) if np.isfinite(score_val) else "N/A"
                    osm_id = info["osm_id"]
                    length_text = f"{info['length']:.1f}" if np.isfinite(info["length"]) else "N/A"
                else:
                    # fallback direct from row (if selection produced an edge not in "best")
                    val = r.get(cost_score)
                    score_text = score_label_fmt.format(float(val)) if pd.notnull(val) else "N/A"
                    osm_id = r.get("osm_id", "N/A")
                    length_val = r.get("length", np.nan)
                    length_text = f"{float(length_val):.1f}" if pd.notnull(length_val) else "N/A"

                tooltip_html = f"""
                <b>OSM ID:</b> {osm_id}<br>
                <b>Length (m):</b> {length_text}<br>
                <b>Score:</b> {score_text}<br>
                """
                _draw_segment_with_label(m, geom_ctx, context_color, tooltip_html, score_text, is_context=True)
                drawn_context_ids.add(idx)

    # styled_table, table_df = display_scores_table(G, paths)
    return m


In [99]:
def plot_paths_to_grid_image(G, paths, origin_point, destination_point, id,
                              path_names=fxn_names, path_colors=colors, path_colors_nicknames=color_nns,
                              grid_size=(3, math.ceil(num_fxns / 3))):

    lat_center = np.mean([origin_point[0], destination_point[0]])
    lon_center = np.mean([origin_point[1], destination_point[1]])

    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")

    driver = webdriver.Chrome(options=chrome_options)
    image_objs = []

    for i, path_info in enumerate(paths):
        path = path_info['paths']
        label_text = path_info['function']

        # get bounds via coordinates
        lat, lon = prepare_path_coordinates(G, path)
        coords = list(zip(lat, lon))

        # include o and d
        all_lats = lat + [origin_point[0], destination_point[0]]
        all_lons = lon + [origin_point[1], destination_point[1]]

        # calc padded bounds
        padding_factor = 0.1
        lat_min, lat_max = min(all_lats), max(all_lats)
        lon_min, lon_max = min(all_lons), max(all_lons)

        lat_pad = (lat_max - lat_min) * padding_factor
        lon_pad = (lon_max - lon_min) * padding_factor

        sw = [lat_min - lat_pad, lon_min - lon_pad]
        ne = [lat_max + lat_pad, lon_max + lon_pad]

        # create map and fit to bounds
        m = folium.Map(zoom_control=False, control_scale=False)
        m.fit_bounds([sw, ne])
        folium.Marker(origin_point, popup="Origin", icon=folium.Icon(color='red')).add_to(m)
        folium.Marker(destination_point, popup="Destination", icon=folium.Icon(color='green')).add_to(m)
        folium.PolyLine(coords, color=path_colors[i % len(path_colors)], weight=5, opacity=1.0).add_to(m)

        with tempfile.NamedTemporaryFile(suffix=".html", delete=False) as f:
            m.save(f.name)
            html_path = f.name

        driver.get("file://" + html_path)
        time.sleep(2)  # wait for map to load

        screenshot_path = html_path + ".png"
        driver.save_screenshot(screenshot_path)
        img = Image.open(screenshot_path)

        draw = ImageDraw.Draw(img)
        try:
            font = ImageFont.truetype("arial.ttf", 100)
        except Exception:
            font = ImageFont.load_default()
        pad = 4
        try:
            left, top, right, bottom = draw.textbbox((0, 0), label_text, font=font)
            text_w, text_h = right - left, bottom - top
        except AttributeError:
            text_w, text_h = draw.textsize(label_text, font=font)
        bg_rect = (0, 0, text_w + 2*pad, text_h + 2*pad)
        draw.rectangle(bg_rect, fill="white")
        draw.text((pad, pad), label_text, fill="black", font=font)

        image_objs.append(img)

        os.remove(html_path)
        os.remove(screenshot_path)

    driver.quit()

    # combine images
    cols, rows = grid_size
    width, height = img.size
    grid_width = cols * width
    grid_height = rows * height
    combined = Image.new('RGB', (grid_width, grid_height), color='white')

    for idx, img in enumerate(image_objs):
        row, col = divmod(idx, cols)
        x = col * width
        y = row * height
        combined.paste(img, (x, y))

    # save images combined
    os.makedirs(f"ranked_paths/{curr_score}/{id}", exist_ok=True)
    combined.save(f"ranked_paths/{curr_score}/{id}/{id}_separated_maps.png")
    print(f"Saved: ranked_paths/{curr_score}/{id}/{id}_separated_maps.png")


In [100]:
def display_scores_table(G, paths, pn_coded = fxn_cols, pn = fxn_names, pn_nick = color_nns):
    
    func_table = []

    # loop through each path and compute all scores
    for path in paths:
        row = []
        # compute scores for each metric
        for func in pn_coded:
            score_info = calculate_path_score(G, path)
            func_score = np.nan if func == 'length' else score_info[2]
            row.append(func_score)
        # append segment count as last column
        num_segments = score_info[1]
        row.append(num_segments)
        func_table.append(row)

    # build df with color labels
    code_to_name = dict(zip(pn_coded, pn))
    cols = [code_to_name[code] for code in pn_coded] + ['segments']
    index_labels = [
        f"{pn[i % len(pn)]} ({pn_nick[i % len(pn_nick)]})"
        for i in range(len(paths))
    ]

    # create df
    func_df = pd.DataFrame(func_table, columns=cols, index=index_labels)

    # table styling for display
    styled = (func_df.style
        # styling for table cells
        .set_properties(**{
            'background-color': 'white',
            'color': 'black'
        })
        # styling for table headers
        .set_table_styles([
            {
                'selector': 'th',
                'props': [
                    ('background-color', 'white'),
                    ('color', 'black'),
                ]
            }
        ])
    )

    # style only the diagonal cells green to indicate optimal score
    for idx, col in enumerate(cols[:-1]):
        row_label = index_labels[idx]
        styled = styled.set_properties(
            subset=pd.IndexSlice[row_label, col],
            **{'color': 'yellowgreen'}
        )

    # can choose to display table or not
    # display(styled)
    return styled, func_df


In [101]:
def table_format_save(table, id):

    # put to html
    table_html = table.to_html()

    # define bg/formtting
    full_html = f"""
    <html>
    <head>
        <style>
            body {{
                background-color: white;
                margin: 0;
                padding: 10px;
                font-family: Arial, sans-serif;
            }}
            table {{
                border-collapse: collapse;
                width: 100%;
            }}
            th, td {{
                padding: 8px 12px;
                border: 1px solid #ccc;
            }}
        </style>
    </head>
    <body>
        {table_html}
    </body>
    </html>
    """

    # save
    hti = Html2Image(output_path=f"sample_paths/{curr_score}/{id}")
    hti.screenshot(
        html_str=full_html,
        save_as=f"{id}_table.png",
        size=(1000, 270)
    )
    return


In [102]:
def routing_summary_to_csv(summary, csv_path=f"{curr_score}_path_summaries.csv"):
    
    # rounding specifications for each col
    rounding = {
        "distance": 2,
        "time_min": 1,
        "normalized_score": 2,
        "inverse_score": 2
    }

    # col names
    columns = ["sample_id", "path_type", "color_name", "distance", "time_min", "normalized_score", "inverse_score", "num_segments"]

    # convert to df
    new_df = pd.DataFrame(summary, columns=columns)

    for col, deci in rounding.items():
        new_df[col] = pd.to_numeric(new_df[col], errors='coerce').round(deci)

    # load existing CSV if exists
    if os.path.exists(csv_path):
        existing_df = pd.read_csv(csv_path)

        # drop existing dupes
        existing_df = existing_df[~(
            existing_df["sample_id"].isin(new_df["sample_id"]) &
            existing_df["path_type"].isin(new_df["path_type"])
        )]

        # add to df
        final_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        final_df = new_df

    # save to CSV
    final_df.to_csv(csv_path, index=False)

In [103]:
def save_map(image, id):
    os.makedirs(f'sample_paths/{curr_score}/{id}', exist_ok=True)
    image.save(f"sample_paths/{curr_score}/{id}/{id}_map.html")
    return

In [104]:


def routing_data_to_csv(df, id, csv_path=f"{curr_score}_table.csv"):
    # chcek output dir
    csv_dir = os.path.dirname(csv_path)
    if csv_dir:
        os.makedirs(csv_dir, exist_ok=True)

    # map row index to type
    path_types = fxn_names

    new_rows = []
    for i, path_type in enumerate(path_types):
        row_data = {
            "sample_id": id,
            "path_type": path_type,
            "Linear 1": df.iloc[i, 0],
            "Linear 1.4": df.iloc[i, 1],
            "Linear 10": df.iloc[i, 2],
            "Inverse": df.iloc[i, 3],
            "Inverse Log": df.iloc[i, 4],
            "segments": df.iloc[i, 6],
        }
        new_rows.append(row_data)

    # make new df
    new_df = pd.DataFrame(new_rows)

    # load existing data if a csv with same path already exists
    if os.path.exists(csv_path):
        existing_df = pd.read_csv(csv_path)

        # remove old rows with dupes
        existing_df = existing_df[~(
            existing_df["sample_id"].isin(new_df["sample_id"]) &
            existing_df["path_type"].isin(new_df["path_type"])
        )]

        final_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        final_df = new_df

    # save
    final_df.to_csv(csv_path, index=False)


<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgb(234, 62, 131);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ do not modify ★ </h3>
  <p> Calling code!</p>
</div>

In [105]:
G, G_2 = prepare_graphs()

In [106]:
def pareto_ordered_paths(G, paths):
    # Count how many other paths dominate each path
    for i, p1 in enumerate(paths):
        domination_count = 0
        for j, p2 in enumerate(paths):
            if i != j:
                if (p2['time'] <= p1['time'] and p2['score'] <= p1['score']) and \
                   (p2['time'] < p1['time'] or p2['score'] < p1['score']):
                    domination_count += 1
        p1['dominated_by'] = domination_count

    # Sort by number of times dominated (ascending)
    paths.sort(key=lambda r: r['dominated_by'])

    
    
    # Optionally remove the extra field
    for p in paths:
        del p['dominated_by']


    return paths


In [111]:
# for i, row in routes_df.iterrows():

#     id = row['sample_id']
#     origin_str = row['origin']
#     destination_str = row['destination']

#     # have to parse
#     origin_lat, origin_lon = map(float, origin_str.split(','))
#     origin_point = (origin_lat, origin_lon)
#     destination_lat, destination_lon = map(float, destination_str.split(','))
#     destination_point = (destination_lat, destination_lon)

#     try:
#         origin_node = ox.nearest_nodes(G_2, origin_lon, origin_lat)
#         destination_node = ox.nearest_nodes(G_2, destination_lon, destination_lat)

#         fxn_vars = []
#         fxn_paths_tested = []
#         for j in range(num_fxns):
#             paths = list(ox.k_shortest_paths(G, origin_node, destination_node, 1, weight=fxn_cols[j]))
#             fxn_vars.append(paths)
#             fxn_paths_tested.append(paths[0])

#         summary, table, df, image = plot_all_paths_in_list(G, fxn_paths_tested, origin_point, destination_point, id)
#         save_map(image, id)
#         routing_data_to_csv(df, id)
#         routing_summary_to_csv(summary)

#         table_format_save(table, id)
#         plot_paths_to_grid_image(G, fxn_paths_tested, origin_point, destination_point, id)


#     except Exception as e:
#         print(f"Skipping route {id}: {e}")
#         continue 
